# Proyecto RaSA — Entendimiento de datos 2
**Entregable 1 — Calidad de datos**

Se revisas las 4 dimensiones de calidad vistas en el curso en las fuentes compartidas por RaSA en la base `RaSaTransaccional` y saque conclusiones.
- **1. Completitud**
- **2. Unicidad**
- **3. Validez**
- **4. Consistencia**

Las fuentes evaluadas son:

- **F1. FuenteAreasDeServicio_Copia_E** — áreas de servicio por condado.
- **F2. FuenteTiposBeneficio_Copia_E** — tipos de beneficio.
- **F3. FuentePlanesBeneficio_Copia_E** — beneficios ofrecidos por cada plan (tabla central).
- **F4. FuenteCondicionesDePago_Copia_E** — condiciones de pago (referencia).

## 0. Configuración y utilidades

In [2]:
# Credenciales y conexión (use su usuario/clave del curso y su servidor de los tutoriales)
db_user = 'DB_202613_cj_hernandezg12'
db_psswd = '201224455'

# Base de datos de las fuentes
source_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/RaSaTransaccional'  # AJUSTE host/puerto al suyo

# Driver JDBC de MySQL
path_jar_driver = r'C:\Users\cjhg0\OneDrive\Documentos\CJHG\MAIT\3.2. Modelado de Datos y ETL\mysql-connector-java-8.0.28.jar'

In [8]:
import os
from pyspark.sql import SparkSession, functions as F

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

spark = (
    SparkSession.builder
    .appName("CalidadDDatos_RaSA")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.jars", path_jar_driver)
    .config("spark.driver.extraClassPath", path_jar_driver)
    .config("spark.executor.extraClassPath", path_jar_driver)
    .getOrCreate()
)

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [9]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    return spark.read.format('jdbc') \
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()

def cargar_tabla(nombre):
    sql = f'(SELECT * FROM RaSaTransaccional.{nombre}) AS t'
    return obtener_dataframe_de_bd(source_db_connection_string, sql, db_user, db_psswd)

In [5]:
# Verifique los nombres reales de las fuentes
tablas_bd = obtener_dataframe_de_bd(source_db_connection_string,
    "(SELECT table_name FROM information_schema.tables WHERE table_schema='RaSaTransaccional' AND table_name LIKE 'Fuente%') AS t",
    db_user, db_psswd)
tablas_bd.show(truncate=False)

TABLAS = {
    'areas':        'FuenteAreasDeServicio_Copia_E',
    'tipos':        'FuenteTiposBeneficio_Copia_E',
    'planes':       'FuentePlanesBeneficio_Copia_E',   # ajuste si en la BD es 'FuentesPlanes...'
    'condiciones':  'FuenteCondicionesDePago_Copia_E',
}

+-------------------------------+
|TABLE_NAME                     |
+-------------------------------+
|FuenteAreasDeServicio_Copia_E  |
|FuenteCondicionesDePago_Copia_E|
|FuentePlanesBeneficio_Copia_E  |
|FuenteTiposBeneficio_Copia_E   |
+-------------------------------+



In [13]:
areas = cargar_tabla(TABLAS['areas'])
areas.show(5)

+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|          Condado|    Estado|PoblacionAct| Area|Densidad|Fecha|
+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|         100622017|New Jersey - Medi...|        34005|Burlington County|New Jersey|    464269.0|805.0|   577.0| 2017|
|         101012018|New Jersey - Medi...|        34031|   Passaic County|New Jersey|    518117.0|185.0|  2801.0| 2018|
|          10132017|BlueOptions16842F...|        12031|     Duval County|   Florida|    999935.0|774.0|  1292.0| 2017|
|         101982018|New Jersey - Medi...|        34003|    Bergen County|New Jersey|    953819.0|234.0|  4076.0| 2018|
|         102012017|New Jersey - Medi...|        34021|    Mercer County|New Jersey|    385898.0|226.0|  1708.0| 2017|
+------------------+--------------------+-------

In [14]:
tipos = cargar_tabla(TABLAS['tipos'])
tipos.show(5)

+-----------------+--------------------+--------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|IdTipoBeneficio_T|              Nombre|     UnidadDelLimite|EsEHB|EstaCubiertaPorSeguro|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|Fecha|
+-----------------+--------------------+--------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|              565|Nutritional Couns...|                    |   No|                  Yes|                     No|                                      No|                                    Yes| 2017|
|              795|Rehabilitative Sp...|Days per Benefit ...|  Yes|                  Yes|                    Yes|                                      No|                                    Yes| 2

In [15]:
planes = cargar_tabla(TABLAS['planes'])
planes.show(5)

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|              640|          10382017|                       34|                         27|                3|16842FL0070128-03|2017-12-31|        16842|          0|           50|            35|
|              225|          31512017|                      238|                         45|                2|29418TX0140002-04|2017-12-31|        29418|          0|            0|          NULL|
|              190|      

In [16]:
condiciones = cargar_tabla(TABLAS['condiciones'])
condiciones.show(5)

+---------------------+--------------------+--------+
|IdCondicionesDePago_T|         Descripcion|    Tipo|
+---------------------+--------------------+--------+
|                  187|Copay with deduct...|  Copago|
|                  204|       Copay per Day|  Copago|
|                   45|         Coinsurance|Coseguro|
|                   85|Copay per Day bef...|  Copago|
|                   18|No Charge after d...|Coseguro|
+---------------------+--------------------+--------+
only showing top 5 rows


## 1. Completitud

La completitud evalúa si los campos requeridos para los análisis de RaSA contienen información suficiente o si presentan valores nulos o vacíos que puedan impedir la integración de fuentes o el cálculo de indicadores.

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

fuentes = {
    "AreasDeServicio": areas,
    "TiposBeneficio": tipos,
    "PlanesBeneficio": planes,
    "CondicionesDePago": condiciones
}

def evaluar_completitud(df, nombre_fuente):
    total_registros = df.count()
    resultados = []

    for columna in df.columns:
        valores_faltantes = df.filter(
            F.col(columna).isNull() |
            (F.trim(F.col(columna).cast("string")) == "")
        ).count()

        porcentaje_faltantes = (
            round((valores_faltantes / total_registros) * 100, 2)
            if total_registros > 0 else 0
        )

        resultados.append((
            nombre_fuente,
            columna,
            total_registros,
            valores_faltantes,
            porcentaje_faltantes
        ))

    return spark.createDataFrame(
        resultados,
        ["Fuente", "Campo", "Total_Registros", "Valores_Faltantes", "Porcentaje_Faltantes"]
    )

In [ ]:
resultados_completitud = []

for nombre_fuente, df in fuentes.items():
    resultados_completitud.append(
        evaluar_completitud(df, nombre_fuente)
    )

completitud = reduce(
    lambda df1, df2: df1.unionByName(df2),
    resultados_completitud
)


In [29]:
completitud_problemas = (
    completitud
    .filter(F.col("Valores_Faltantes") > 0)
    .orderBy(F.desc("Porcentaje_Faltantes"))
)

completitud_problemas.show(100, truncate=False)

+---------------+------------------+---------------+-----------------+--------------------+
|Fuente         |Campo             |Total_Registros|Valores_Faltantes|Porcentaje_Faltantes|
+---------------+------------------+---------------+-----------------+--------------------+
|PlanesBeneficio|cantidadLimite    |36036          |30571            |84.83               |
|TiposBeneficio |UnidadDelLimite   |849            |559              |65.84               |
|PlanesBeneficio|IdTipoBeneficio_T |36036          |2086             |5.79                |
|PlanesBeneficio|IdPlan_T          |36036          |2053             |5.7                 |
|PlanesBeneficio|IdAreaDeServicio_T|36036          |2041             |5.66                |
|AreasDeServicio|IdGeografia_T     |188815         |6378             |3.38                |
|AreasDeServicio|IdAreaDeServicio_T|188815         |6288             |3.33                |
|AreasDeServicio|Densidad          |188815         |2407             |1.27      

In [30]:
resumen_completitud = (
    completitud
    .groupBy("Fuente")
    .agg(
        F.count("Campo").alias("Total_Campos"),
        F.sum(F.when(F.col("Valores_Faltantes") > 0, 1).otherwise(0)).alias("Campos_Con_Faltantes"),
        F.sum("Valores_Faltantes").alias("Total_Valores_Faltantes"),
        F.max("Porcentaje_Faltantes").alias("Mayor_Porcentaje_Faltantes")
    )
    .orderBy("Fuente")
)

resumen_completitud.show(truncate=False)

+-----------------+------------+--------------------+-----------------------+--------------------------+
|Fuente           |Total_Campos|Campos_Con_Faltantes|Total_Valores_Faltantes|Mayor_Porcentaje_Faltantes|
+-----------------+------------+--------------------+-----------------------+--------------------------+
|AreasDeServicio  |9           |4                   |17480                  |3.38                      |
|CondicionesDePago|3           |0                   |0                      |0.0                       |
|PlanesBeneficio  |11          |4                   |36751                  |84.83                     |
|TiposBeneficio   |9           |1                   |559                    |65.84                     |
+-----------------+------------+--------------------+-----------------------+--------------------------+



### Análisis de completitud
El análisis de completitud confirma los hallazgos identificados durante el perfilamiento inicial y permite evaluar su impacto sobre la calidad de los datos. Aunque algunas fuentes presentan un buen nivel de completitud, existen campos faltantes que afectan directamente la integración entre tablas y la construcción de análisis confiables.

La fuente `CondicionesDePago` no presenta valores faltantes en los campos evaluados, por lo que no se identifican problemas de completitud en esta fuente. En contraste, `PlanesBeneficio` concentra el mayor volumen de faltantes, principalmente en `cantidadLimite`, con 30.571 registros sin información, equivalentes al 84,83%. Este hallazgo debe analizarse junto con la regla de negocio de beneficios con límite cuantitativo, ya que no todos los beneficios necesariamente requieren una cantidad límite.

En `TiposBeneficio`, el campo `UnidadDelLimite` presenta 559 valores faltantes, equivalentes al 65,84%. Este resultado también debe interpretarse en relación con el atributo `TieneLimiteCuantitativo`, dado que la unidad del límite solo sería obligatoria para beneficios que manejan límites cuantitativos.

Por su parte, `AreasDeServicio` presenta faltantes en `IdGeografia_T`, `IdAreaDeServicio_T`, `Area` y `Densidad`. Aunque los porcentajes son bajos, entre 1,27% y 3,38%, el faltante en `IdAreaDeServicio_T` es especialmente crítico porque este campo permite relacionar las áreas de servicio con la fuente central de planes-beneficio.

En conclusión, la completitud de los datos es parcialmente adecuada para continuar con el proyecto. Sin embargo, antes de construir el modelo dimensional deben definirse reglas para tratar los campos faltantes, especialmente aquellos asociados con llaves de integración, variables geográficas y atributos requeridos para validar límites cuantitativos.

## 2. Unicidad

La unicidad evalúa si los registros o identificadores permiten reconocer de manera única cada entidad dentro de una fuente. En esta sección se revisa la cantidad de registros totales, registros completamente únicos, registros duplicados y valores distintos de las llaves principales identificadas en cada fuente.

Este análisis permite identificar posibles duplicidades, presencia de información histórica o problemas en la definición de llaves operacionales.

In [31]:
def evaluar_unicidad(df, nombre_fuente, llave):
    total_registros = df.count()
    registros_unicos = df.distinct().count()
    duplicados_completos = total_registros - registros_unicos

    valores_distintos_llave = df.select(llave).distinct().count()

    llaves_repetidas = (
        df
        .groupBy(llave)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    porcentaje_duplicados_completos = (
        round((duplicados_completos / total_registros) * 100, 2)
        if total_registros > 0 else 0
    )

    return spark.createDataFrame(
        [(
            nombre_fuente,
            llave,
            total_registros,
            registros_unicos,
            duplicados_completos,
            porcentaje_duplicados_completos,
            valores_distintos_llave,
            llaves_repetidas
        )],
        [
            "Fuente",
            "Llave_Evaluada",
            "Total_Registros",
            "Registros_Unicos",
            "Duplicados_Completos",
            "Porcentaje_Duplicados_Completos",
            "Valores_Distintos_Llave",
            "Llaves_Repetidas"
        ]
    )

In [32]:
unicidad_areas = evaluar_unicidad(
    areas,
    "AreasDeServicio",
    "IdAreaDeServicio_T"
)

unicidad_tipos = evaluar_unicidad(
    tipos,
    "TiposBeneficio",
    "IdTipoBeneficio_T"
)

unicidad_planes = evaluar_unicidad(
    planes,
    "PlanesBeneficio",
    "IdPlan_T"
)

unicidad_condiciones = evaluar_unicidad(
    condiciones,
    "CondicionesDePago",
    "IdCondicionesDePago_T"
)

In [33]:
unicidad = (
    unicidad_areas
    .unionByName(unicidad_tipos)
    .unionByName(unicidad_planes)
    .unionByName(unicidad_condiciones)
)

unicidad.show(truncate=False)

+-----------------+---------------------+---------------+----------------+--------------------+-------------------------------+-----------------------+----------------+
|Fuente           |Llave_Evaluada       |Total_Registros|Registros_Unicos|Duplicados_Completos|Porcentaje_Duplicados_Completos|Valores_Distintos_Llave|Llaves_Repetidas|
+-----------------+---------------------+---------------+----------------+--------------------+-------------------------------+-----------------------+----------------+
|AreasDeServicio  |IdAreaDeServicio_T   |188815         |145242          |43573               |23.08                          |5410                   |5384            |
|TiposBeneficio   |IdTipoBeneficio_T    |849            |578             |271                 |31.92                          |178                    |132             |
|PlanesBeneficio  |IdPlan_T             |36036          |27543           |8493                |23.57                          |393                    |392 

La tabla anterior resume el comportamiento de unicidad de las cuatro fuentes. Se observa que ninguna de las llaves evaluadas identifica de forma única cada fila, ya que todas presentan llaves repetidas y registros duplicados completos.

En `AreasDeServicio`, `TiposBeneficio` y `CondicionesDePago`, la repetición de identificadores puede representar duplicidad operacional, registros históricos o variaciones en atributos descriptivos para una misma entidad.

En `PlanesBeneficio`, la repetición de `IdPlan_T` debe interpretarse con cuidado, ya que un mismo plan puede estar asociado a varios beneficios, áreas de servicio, proveedores, niveles de servicio y fechas. Por esta razón, se realiza una validación adicional con una posible llave compuesta.

In [ ]:
# Validación opcional: ejemplos de llaves repetidas por fuente.
# Se deja como soporte técnico, pero no se usa como evidencia principal del análisis.

areas_repetidas = (
    areas
    .groupBy("IdAreaDeServicio_T")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

#areas_repetidas.show(20, truncate=False)

tipos_repetidos = (
    tipos
    .groupBy("IdTipoBeneficio_T")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

#tipos_repetidos.show(20, truncate=False)

condiciones_repetidas = (
    condiciones
    .groupBy("IdCondicionesDePago_T")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

#condiciones_repetidas.show(20, truncate=False)

planes_repetidos = (
    planes
    .groupBy("IdPlan_T")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

#planes_repetidos.show(20, truncate=False)


### Validación adicional de unicidad en PlanesBeneficio

Dado que `IdPlan_T` no representa por sí solo la granularidad de la fuente `PlanesBeneficio`, se evalúa una posible llave compuesta conformada por plan, tipo de beneficio, área de servicio, proveedor y fecha.

In [40]:
llave_compuesta_planes = [
    "IdPlan_T",
    "IdTipoBeneficio_T",
    "IdAreaDeServicio_T",
    "IdProveedor_T",
    "Fecha"
]

total_planes = planes.count()

planes_unicos_llave_compuesta = (
    planes
    .select(llave_compuesta_planes)
    .distinct()
    .count()
)

duplicados_llave_compuesta = total_planes - planes_unicos_llave_compuesta

print("Total registros PlanesBeneficio:", total_planes)
print("Registros únicos según llave compuesta:", planes_unicos_llave_compuesta)
print("Duplicados según llave compuesta:", duplicados_llave_compuesta)
print("Porcentaje duplicados llave compuesta:", round((duplicados_llave_compuesta / total_planes) * 100, 2))

Total registros PlanesBeneficio: 36036
Registros únicos según llave compuesta: 18682
Duplicados según llave compuesta: 17354
Porcentaje duplicados llave compuesta: 48.16


In [41]:
planes_duplicados_llave_compuesta = (
    planes
    .groupBy(llave_compuesta_planes)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

planes_duplicados_llave_compuesta.show(20, truncate=False)

+-----------------+-----------------+------------------+-------------+----------+-----+
|IdPlan_T         |IdTipoBeneficio_T|IdAreaDeServicio_T|IdProveedor_T|Fecha     |count|
+-----------------+-----------------+------------------+-------------+----------+-----+
|38345WI0080018-03|NULL             |50152017          |38345        |2017-12-31|28   |
|87571OK0350027-05|NULL             |95082017          |87571        |2017-12-31|25   |
|67243LA0100009-01|NULL             |82642017          |67243        |2017-12-31|22   |
|33602TX0460570-01|NULL             |37932018          |33602        |2018-12-31|20   |
|37833WI0380063-01|NULL             |45922017          |37833        |2017-12-31|20   |
|36194FL0130003-01|NULL             |44112018          |36194        |2018-12-31|16   |
|37833WI0540010-01|NULL             |46912018          |37833        |2018-12-31|16   |
|37833WI0540010-01|NULL             |46922017          |37833        |2017-12-31|15   |
|67243LA0090013-05|NULL         

### Análisis de unicidad

El análisis de unicidad evidencia que ninguna de las llaves evaluadas identifica de forma única cada fila de su respectiva fuente. Todas las fuentes presentan registros duplicados completos y repetición de identificadores, lo cual confirma que existen problemas de duplicidad o que algunas fuentes contienen información histórica o de detalle que no puede ser identificada únicamente con una llave simple.

En `AreasDeServicio`, la fuente contiene 188.815 registros, pero solo 5.410 valores distintos de `IdAreaDeServicio_T`. Este resultado confirma que el identificador del área de servicio se repite múltiples veces dentro de la tabla. Además, se identifican 43.573 registros duplicados completos, equivalentes al 23,08% del total. La diferencia frente a los 5.409 valores esperados por el negocio puede explicarse por la existencia de valores nulos en la llave, identificados previamente en el análisis de completitud.

En `TiposBeneficio`, se observan 849 registros, 578 registros únicos y 271 duplicados completos, equivalentes al 31,92%. Adicionalmente, la llave `IdTipoBeneficio_T` presenta 178 valores distintos y 132 identificadores repetidos. Esto indica que un mismo tipo de beneficio aparece más de una vez, posiblemente por variaciones temporales, diferencias en atributos descriptivos o duplicidad operacional.

En `CondicionesDePago`, aunque la fuente tiene un volumen bajo de registros, también se identifican problemas de unicidad. De 31 registros, solo 24 son únicos y existen 7 duplicados completos, equivalentes al 22,58%. La llave `IdCondicionesDePago_T` presenta 23 valores distintos y 7 identificadores repetidos, por lo que se requiere depuración antes de usar esta fuente como dimensión de referencia.

En `PlanesBeneficio`, la repetición de `IdPlan_T` debe interpretarse de manera diferente. La fuente contiene 36.036 registros y solo 393 planes distintos, pero esto no necesariamente representa un error, ya que un mismo plan puede estar asociado a múltiples beneficios, áreas de servicio, proveedores, niveles de servicio y fechas. Sin embargo, se identifican 8.493 duplicados completos, equivalentes al 23,57%, lo cual sí sugiere que existen registros exactamente repetidos que deberían ser revisados o eliminados antes de construir el modelo dimensional.

En conclusión, la unicidad de los datos no es completamente adecuada para el modelado analítico sin aplicar reglas de depuración. Se recomienda eliminar duplicados completos, revisar la definición de llaves para cada fuente y, especialmente en `PlanesBeneficio`, utilizar una llave compuesta que represente mejor el nivel de detalle de la tabla.

## 3. Validez

La validez evalúa si los valores registrados cumplen con los formatos, dominios y rangos esperados según la naturaleza de cada campo y las reglas de negocio conocidas. 

En esta sección se revisan valores fuera de rango, fechas no válidas, dominios no esperados y valores que pueden afectar la interpretación de los datos antes de construir el modelo dimensional.

In [59]:
resultados_validez = []

# AreasDeServicio: Area no debería ser negativa ni cero
area_invalida = areas.filter(
    (F.col("Area").isNotNull()) & (F.col("Area") <= 0)
).count()

resultados_validez.append((
    "AreasDeServicio",
    "Area",
    "El área debe ser mayor que cero",
    area_invalida
))

# AreasDeServicio: PoblacionAct no debería ser negativa
poblacion_invalida = areas.filter(
    (F.col("PoblacionAct").isNotNull()) & (F.col("PoblacionAct") < 0)
).count()

resultados_validez.append((
    "AreasDeServicio",
    "PoblacionAct",
    "La población no debe ser negativa",
    poblacion_invalida
))

# AreasDeServicio: Densidad no debería ser negativa
densidad_invalida = areas.filter(
    (F.col("Densidad").isNotNull()) & (F.col("Densidad") < 0)
).count()

resultados_validez.append((
    "AreasDeServicio",
    "Densidad",
    "La densidad no debe ser negativa",
    densidad_invalida
))

# PlanesBeneficio: valorCopago no debería ser negativo
copago_negativo = planes.filter(
    (F.col("valorCopago").isNotNull()) & (F.col("valorCopago") < 0)
).count()

resultados_validez.append((
    "PlanesBeneficio",
    "valorCopago",
    "El valor de copago no debe ser negativo",
    copago_negativo
))

# PlanesBeneficio: valorCoseguro no debería ser negativo
coseguro_negativo = planes.filter(
    (F.col("valorCoseguro").isNotNull()) & (F.col("valorCoseguro") < 0)
).count()

resultados_validez.append((
    "PlanesBeneficio",
    "valorCoseguro",
    "El valor de coseguro no debe ser negativo",
    coseguro_negativo
))

# PlanesBeneficio: cantidadLimite no debería ser negativa
cantidad_limite_negativa = planes.filter(
    (F.col("cantidadLimite").isNotNull()) & (F.col("cantidadLimite") < 0)
).count()

resultados_validez.append((
    "PlanesBeneficio",
    "cantidadLimite",
    "La cantidad límite no debe ser negativa",
    cantidad_limite_negativa
))

In [60]:
def extraer_anio_fecha(columna):
    return F.coalesce(
        F.when(F.col(columna).cast("string").rlike("^[0-9]{4}$"), F.col(columna).cast("int")),
        F.year(F.try_to_timestamp(F.col(columna).cast("string"), F.lit("yyyy-MM-dd"))),
        F.year(F.try_to_timestamp(F.col(columna).cast("string"), F.lit("yyyy-MM-dd HH:mm:ss"))),
        F.year(F.try_to_timestamp(F.col(columna).cast("string"), F.lit("MMM dd,yyyy"))),
        F.year(F.try_to_timestamp(F.col(columna).cast("string"), F.lit("MMM dd, yyyy")))
    )

areas_con_anio = areas.withColumn("Anio", extraer_anio_fecha("Fecha"))

anios_areas_invalidos = areas_con_anio.filter(
    (F.col("Anio").isNull()) | (~F.col("Anio").between(2017, 2019))
).count()

resultados_validez.append((
    "AreasDeServicio",
    "Fecha",
    "El año debe estar entre 2017 y 2019",
    anios_areas_invalidos
))

tipos_con_anio = tipos.withColumn("Anio", extraer_anio_fecha("Fecha"))

anios_tipos_invalidos = tipos_con_anio.filter(
    (F.col("Anio").isNull()) | (~F.col("Anio").between(2017, 2019))
).count()

resultados_validez.append((
    "TiposBeneficio",
    "Fecha",
    "El año debe estar entre 2017 y 2019",
    anios_tipos_invalidos
))

planes_con_anio = planes.withColumn("Anio", extraer_anio_fecha("Fecha"))

anios_planes_invalidos = planes_con_anio.filter(
    (F.col("Anio").isNull()) | (~F.col("Anio").between(2017, 2019))
).count()

resultados_validez.append((
    "PlanesBeneficio",
    "Fecha",
    "El año de la fecha debe estar entre 2017 y 2019",
    anios_planes_invalidos
))

In [61]:
campos_bandera_tipos = [
    "EsEHB",
    "EstaCubiertaPorSeguro",
    "TieneLimiteCuantitativo",
    "ExcluidoDelDesembolsoMaximoDentroDeLaRed",
    "ExcluidoDelDesembolsoMaximoFueraDeLaRed"
]

valores_validos_bandera = ["Yes", "No", "True", "False", "Si", "Sí", "Nein"]

for campo in campos_bandera_tipos:
    valores_invalidos = (
        tipos
        .filter(~F.col(campo).isin(valores_validos_bandera))
        .count()
    )

    resultados_validez.append((
        "TiposBeneficio",
        campo,
        "El campo debe tener valores booleanos reconocibles",
        valores_invalidos
    ))

In [62]:
valores_validos_tipo_condicion = ["Copago", "Coseguro", "Anticipado"]

condiciones_tipo_invalidos = condiciones.filter(
    ~F.col("Tipo").isin(valores_validos_tipo_condicion)
).count()

resultados_validez.append((
    "CondicionesDePago",
    "Tipo",
    "El tipo de condición debe pertenecer al dominio esperado",
    condiciones_tipo_invalidos
))

In [63]:
validez = spark.createDataFrame(
    resultados_validez,
    [
        "Fuente",
        "Campo",
        "Regla_Validez",
        "Registros_Invalidos"
    ]
)

validez_problemas = (
    validez
    .filter(F.col("Registros_Invalidos") > 0)
    .orderBy(F.desc("Registros_Invalidos"))
)

validez_problemas.show(100, truncate=False)

+-----------------+----------------------------------------+--------------------------------------------------------+-------------------+
|Fuente           |Campo                                   |Regla_Validez                                           |Registros_Invalidos|
+-----------------+----------------------------------------+--------------------------------------------------------+-------------------+
|AreasDeServicio  |Fecha                                   |El año debe estar entre 2017 y 2019                     |6282               |
|AreasDeServicio  |Area                                    |El área debe ser mayor que cero                         |6177               |
|PlanesBeneficio  |Fecha                                   |El año de la fecha debe estar entre 2017 y 2019         |2052               |
|CondicionesDePago|Tipo                                    |El tipo de condición debe pertenecer al dominio esperado|5                  |
|TiposBeneficio   |ExcluidoDelDese

### Conclusión de validez

El análisis de validez evidencia que algunas fuentes contienen valores que no cumplen con los rangos, formatos o dominios esperados para los campos evaluados. Estos hallazgos son relevantes porque pueden afectar la confiabilidad de los análisis geográficos, temporales y de clasificación requeridos por RaSA.

En la fuente `AreasDeServicio` se identifican dos problemas principales. El primero corresponde al campo `Fecha`, donde se encontraron 6.282 registros con años fuera del rango esperado de 2017 a 2019. Este resultado confirma la presencia de valores temporales inválidos, como el año 1800 identificado previamente en el perfilamiento. El segundo problema corresponde al campo `Area`, con 6.177 registros inválidos bajo la regla de que el área debe ser mayor que cero. Este hallazgo es crítico porque el área territorial no debería tener valores negativos o iguales a cero, y puede afectar análisis relacionados con cobertura, densidad y concentración de planes.

En la fuente `PlanesBeneficio` se identificaron 2.052 registros con fechas fuera del rango esperado de 2017 a 2019. Dado que esta fuente es central para analizar la oferta de planes, beneficios, proveedores y áreas de servicio, los problemas de validez temporal pueden afectar los análisis de evolución de planes y comparación entre años.

En la fuente `CondicionesDePago` se encontraron 5 registros cuyo campo `Tipo` no pertenece al dominio esperado. Esto puede afectar la clasificación correcta de condiciones de copago, coseguro o pago anticipado, y generar inconsistencias al relacionar esta fuente con `PlanesBeneficio`.

Por último, en `TiposBeneficio` se identificaron 2 registros inválidos en el campo `ExcluidoDelDesembolsoMaximoDentroDeLaRed`, debido a valores que no corresponden a dominios booleanos reconocibles. Aunque el volumen de registros afectados es bajo, este tipo de inconsistencias debe corregirse para evitar errores en filtros, segmentaciones o reglas de negocio relacionadas con la cobertura de beneficios.

En conclusión, la validez de los datos es parcialmente adecuada para continuar con el proyecto. Sin embargo, antes de construir el modelo dimensional se recomienda corregir o excluir registros con fechas inválidas, revisar los valores no válidos en campos geográficos, estandarizar dominios categóricos y homologar los campos tipo bandera. Estas acciones permitirán mejorar la confiabilidad de los análisis solicitados por RaSA.

## 4. Consistencia

La consistencia evalúa si los datos son coherentes con las reglas de negocio conocidas y si las relaciones entre fuentes pueden construirse correctamente. 

En esta sección se validan conteos esperados por RaSA, correspondencia entre llaves de integración y reglas de negocio entre fuentes, con el objetivo de identificar inconsistencias que puedan afectar el modelo dimensional y los análisis solicitados.

In [64]:
resultados_consistencia = []

# 1. Regla de negocio: RaSA reporta 5.409 áreas de servicio
areas_distintas = (
    areas
    .filter(F.col("IdAreaDeServicio_T").isNotNull())
    .select("IdAreaDeServicio_T")
    .distinct()
    .count()
)

resultados_consistencia.append((
    "AreasDeServicio",
    "Cantidad de áreas de servicio",
    "La fuente debe contener 5.409 áreas de servicio distintas",
    "5409",
    str(areas_distintas),
    0 if areas_distintas == 5409 else abs(areas_distintas - 5409),
    "Cumple" if areas_distintas == 5409 else "No cumple"
))


# 2. Regla de negocio: RaSA reporta 170 tipos de beneficio
tipos_distintos = (
    tipos
    .filter(F.col("IdTipoBeneficio_T").isNotNull())
    .select("IdTipoBeneficio_T")
    .distinct()
    .count()
)

resultados_consistencia.append((
    "TiposBeneficio",
    "Cantidad de tipos de beneficio",
    "La fuente debe contener 170 tipos de beneficio distintos",
    "170",
    str(tipos_distintos),
    0 if tipos_distintos == 170 else abs(tipos_distintos - 170),
    "Cumple" if tipos_distintos == 170 else "No cumple"
))


# 3. Regla de negocio: 15 condiciones de copago
condiciones_copago = (
    condiciones
    .filter(F.trim(F.col("Tipo")) == "Copago")
    .select("IdCondicionesDePago_T")
    .distinct()
    .count()
)

resultados_consistencia.append((
    "CondicionesDePago",
    "Cantidad de condiciones de copago",
    "La fuente debe contener 15 condiciones de copago",
    "15",
    str(condiciones_copago),
    0 if condiciones_copago == 15 else abs(condiciones_copago - 15),
    "Cumple" if condiciones_copago == 15 else "No cumple"
))


# 4. Regla de negocio: 5 condiciones de coseguro
condiciones_coseguro = (
    condiciones
    .filter(F.trim(F.col("Tipo")) == "Coseguro")
    .select("IdCondicionesDePago_T")
    .distinct()
    .count()
)

resultados_consistencia.append((
    "CondicionesDePago",
    "Cantidad de condiciones de coseguro",
    "La fuente debe contener 5 condiciones de coseguro",
    "5",
    str(condiciones_coseguro),
    0 if condiciones_coseguro == 5 else abs(condiciones_coseguro - 5),
    "Cumple" if condiciones_coseguro == 5 else "No cumple"
))

In [65]:
# Catálogos de referencia sin nulos
areas_ref = (
    areas
    .filter(F.col("IdAreaDeServicio_T").isNotNull())
    .select("IdAreaDeServicio_T")
    .distinct()
)

tipos_ref = (
    tipos
    .filter(F.col("IdTipoBeneficio_T").isNotNull())
    .select("IdTipoBeneficio_T")
    .distinct()
)

condiciones_ref = (
    condiciones
    .filter(F.col("IdCondicionesDePago_T").isNotNull())
    .select("IdCondicionesDePago_T")
    .distinct()
)

In [66]:
# Identificar tipos de beneficio que tienen límite cuantitativo
tipos_con_limite = (
    tipos
    .filter(
        F.lower(F.trim(F.col("TieneLimiteCuantitativo").cast("string")))
        .isin("yes", "si", "sí", "true")
    )
    .select("IdTipoBeneficio_T")
    .distinct()
)

# Planes asociados a tipos de beneficio con límite cuantitativo
planes_con_tipos_limite = (
    planes
    .join(
        tipos_con_limite,
        "IdTipoBeneficio_T",
        "inner"
    )
)

# Registros que deberían tener cantidad límite, pero no la tienen o es inválida
planes_limite_inconsistente = (
    planes_con_tipos_limite
    .filter(
        F.col("cantidadLimite").isNull() |
        (F.col("cantidadLimite") <= 0)
    )
    .count()
)

resultados_consistencia.append((
    "PlanesBeneficio - TiposBeneficio",
    "Consistencia de límite cuantitativo",
    "Si un beneficio tiene límite cuantitativo, cantidadLimite debe ser mayor que cero",
    "0 registros inconsistentes",
    str(planes_limite_inconsistente),
    planes_limite_inconsistente,
    "Cumple" if planes_limite_inconsistente == 0 else "No cumple"
))

In [67]:
consistencia = spark.createDataFrame(
    resultados_consistencia,
    [
        "Fuente",
        "Validacion",
        "Regla_Consistencia",
        "Valor_Esperado",
        "Valor_Encontrado",
        "Registros_Problema",
        "Estado"
    ]
).dropDuplicates()

consistencia.show(100, truncate=False)

+--------------------------------+-----------------------------------+---------------------------------------------------------------------------------+--------------------------+----------------+------------------+---------+
|Fuente                          |Validacion                         |Regla_Consistencia                                                               |Valor_Esperado            |Valor_Encontrado|Registros_Problema|Estado   |
+--------------------------------+-----------------------------------+---------------------------------------------------------------------------------+--------------------------+----------------+------------------+---------+
|AreasDeServicio                 |Cantidad de áreas de servicio      |La fuente debe contener 5.409 áreas de servicio distintas                        |5409                      |5409            |0                 |Cumple   |
|TiposBeneficio                  |Cantidad de tipos de beneficio     |La fuente debe contener 17

In [68]:
consistencia_problemas = (
    consistencia
    .filter(F.col("Estado") == "No cumple")
    .orderBy(F.desc("Registros_Problema"))
)

consistencia_problemas.show(100, truncate=False)

+--------------------------------+-----------------------------------+---------------------------------------------------------------------------------+--------------------------+----------------+------------------+---------+
|Fuente                          |Validacion                         |Regla_Consistencia                                                               |Valor_Esperado            |Valor_Encontrado|Registros_Problema|Estado   |
+--------------------------------+-----------------------------------+---------------------------------------------------------------------------------+--------------------------+----------------+------------------+---------+
|PlanesBeneficio - TiposBeneficio|Consistencia de límite cuantitativo|Si un beneficio tiene límite cuantitativo, cantidadLimite debe ser mayor que cero|0 registros inconsistentes|14471           |14471             |No cumple|
|TiposBeneficio                  |Cantidad de tipos de beneficio     |La fuente debe contener 17

### Conclusión de consistencia

El análisis de consistencia evidencia que algunas fuentes no cumplen completamente con las reglas de negocio definidas para el proyecto. Aunque varias validaciones de integración entre fuentes no presentan problemas, se identifican inconsistencias relevantes en la relación entre `PlanesBeneficio` y `TiposBeneficio`, así como en los conteos esperados de tipos de beneficio y condiciones de pago.

El hallazgo más crítico se presenta en la validación de límite cuantitativo. Se identificaron 14.471 registros en `PlanesBeneficio` asociados a tipos de beneficio marcados con límite cuantitativo, pero cuya `cantidadLimite` es nula, cero o inválida. Esta inconsistencia es relevante porque afecta directamente los análisis relacionados con beneficios, condiciones de cobertura y evolución de planes, ya que no sería posible interpretar correctamente los límites aplicables para esos registros.

Adicionalmente, la fuente `TiposBeneficio` contiene 178 tipos de beneficio distintos, mientras que la regla de negocio indica que deberían existir 170. Esta diferencia de 8 tipos sugiere la presencia de registros adicionales, duplicidades lógicas, diferencias históricas o valores que requieren validación con el negocio antes de ser utilizados como catálogo definitivo.

En la fuente `CondicionesDePago`, se identificaron 14 condiciones de copago, frente a las 15 esperadas por negocio. Aunque la diferencia es de solo un registro, este hallazgo puede afectar la clasificación de condiciones de pago y la correcta interpretación de los planes que hacen referencia a copagos.

En conclusión, la consistencia de los datos es parcial. Antes de construir el modelo dimensional, se recomienda revisar con RaSA las diferencias frente a los conteos esperados, depurar los catálogos de tipos de beneficio y condiciones de pago, y definir una regla clara para los registros con límite cuantitativo que no cuentan con `cantidadLimite` válida. Estas acciones son necesarias para evitar inconsistencias en los análisis de cobertura, costos y evolución de los planes.

## Conclusiones generales del análisis de calidad de datos

A partir del análisis de calidad realizado sobre las fuentes compartidas por RaSA, se concluye que los datos contienen información relevante para responder los análisis solicitados; sin embargo, presentan problemas de calidad que deben ser tratados antes de avanzar hacia la construcción del modelo dimensional y los tableros de control.

En términos de **completitud**, se evidenció que algunas fuentes presentan valores faltantes en campos clave. La fuente `CondicionesDePago` no presentó problemas de completitud, mientras que `PlanesBeneficio`, `TiposBeneficio` y `AreasDeServicio` sí contienen campos con información ausente. Los casos más relevantes se encuentran en `cantidadLimite`, `UnidadDelLimite` y en identificadores como `IdAreaDeServicio_T`, ya que estos campos son necesarios para interpretar límites de beneficios e integrar correctamente las fuentes.

Respecto a la **unicidad**, se identificó que las llaves operacionales evaluadas no identifican de manera única cada fila en las fuentes. Se encontraron registros duplicados completos y llaves repetidas en todas las tablas. En particular, la fuente `PlanesBeneficio` requiere un análisis especial, ya que `IdPlan_T` no representa por sí solo la granularidad de la tabla. Esto indica que, para el modelado analítico, será necesario eliminar duplicados completos y definir llaves compuestas o identificadores adecuados según el nivel de detalle de cada fuente.

En cuanto a la **validez**, se encontraron valores que no cumplen con los rangos, formatos o dominios esperados. Entre los principales hallazgos se encuentran fechas fuera del rango esperado, áreas inválidas, valores no reconocidos en campos categóricos y errores en campos tipo bandera. Estos problemas pueden afectar análisis geográficos, temporales y de clasificación, por lo que se requiere aplicar reglas de limpieza, estandarización y validación antes de utilizar los datos en procesos analíticos.

Frente a la **consistencia**, se evidenciaron diferencias entre los datos y algunas reglas de negocio definidas por RaSA. Se identificaron inconsistencias en el número de tipos de beneficio, en la cantidad de condiciones de copago y, especialmente, en registros de `PlanesBeneficio` asociados a beneficios con límite cuantitativo que no cuentan con una `cantidadLimite` válida. Este último hallazgo es crítico, ya que puede afectar directamente el análisis de beneficios, cobertura y condiciones ofrecidas por los planes.

En conclusión, la calidad de los datos es **parcialmente adecuada** para continuar con el proyecto, siempre que se incorporen reglas de tratamiento previas al modelado dimensional. Las acciones recomendadas incluyen eliminar duplicados completos, tratar valores faltantes en llaves y campos críticos, estandarizar dominios categóricos, corregir fechas inválidas, validar valores fuera de rango y revisar con el negocio las diferencias encontradas frente a las reglas esperadas.

Finalmente, se considera viable avanzar hacia las siguientes etapas del proyecto, pero dejando documentados los supuestos, restricciones y decisiones de limpieza que se apliquen. Esto permitirá que el almacén de datos y los tableros resultantes reflejen información más confiable, consistente y útil para los análisis requeridos por RaSA.